# Project Homework: Applying GitHub Ingestion to OWASP LLM Top 10

This notebook applies the `read_repo_data()` pattern learned in the course to a different repository structure. The OWASP LLM Top 10 repository provides security documentation for large language model applications.

## What's Different

Unlike the course repositories (DataTalks FAQ and Evidently docs):
- **OWASP has NO frontmatter** - files are plain markdown without YAML metadata blocks
- **Different organization** - nested structure with 2_0_vulns/, documentation/, translations/ folders
- **More files** - 542 markdown documents vs 95-1285 in course repos

## Key Learning

The same `read_repo_data()` implementation works across all repositories because:
1. python-frontmatter gracefully handles files without frontmatter (returns empty metadata dict)
2. In-memory zip processing works regardless of repository structure
3. Markdown filtering works uniformly across file organizations

In [1]:
# Import required libraries
import requests
from zipfile import ZipFile
from io import BytesIO
import frontmatter
from pathlib import Path
from typing import List, Dict, Any

## Helper Functions

These functions are copied from course/day1.ipynb (per learning objective - adaptation through reuse, not reinvention). Each function has been enhanced with type hints and docstrings per project/ engineering standards.

In [2]:
def download_repo_zip(repo_owner: str, repo_name: str) -> bytes:
    """Download GitHub repository as zip archive.

    Uses GitHub's codeload API which provides repositories as zip archives
    without requiring authentication for public repos.

    Args:
        repo_owner: GitHub username or organization (e.g., 'OWASP')
        repo_name: Repository name (e.g., 'www-project-top-10-for-large-language-model-applications')

    Returns:
        Binary content of zip archive

    Raises:
        requests.HTTPError: If download fails (repo not found, network error, etc.)
    """
    url = f"https://codeload.github.com/{repo_owner}/{repo_name}/zip/refs/heads/main"
    print(f"Downloading {repo_owner}/{repo_name} from {url}")

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    print(f"Downloaded {len(response.content)} bytes")
    return response.content

In [3]:
def is_markdown_file(filename: str) -> bool:
    """Check if file is a markdown document (.md or .mdx).

    Args:
        filename: Full path to file in zip archive

    Returns:
        True if file ends with .md or .mdx, False otherwise
    """
    path = Path(filename)
    return path.suffix in ['.md', '.mdx']

In [4]:
def parse_markdown_with_frontmatter(content_bytes: bytes) -> Dict[str, Any]:
    """Parse markdown file extracting frontmatter metadata and content.

    Handles files without frontmatter gracefully - returns empty metadata dict.
    This is critical for OWASP repository which has no frontmatter.

    Args:
        content_bytes: Raw file content as bytes

    Returns:
        Dictionary with:
        - 'metadata': dict of frontmatter key-value pairs (empty {} if no frontmatter)
        - 'content': str of markdown content after frontmatter (or full content if none)
    """
    content_str = content_bytes.decode('utf-8', errors='ignore')
    post = frontmatter.loads(content_str)

    return {
        'metadata': dict(post.metadata),  # Empty dict {} if no frontmatter
        'content': post.content
    }

## Main Function: read_repo_data()

This is the same orchestration logic from the course, proving that good abstractions work across different repository structures.

In [5]:
def read_repo_data(repo_owner: str, repo_name: str) -> List[Dict[str, Any]]:
    """Download GitHub repository and extract markdown documentation with metadata.

    This implementation is identical to course/day1.ipynb because the core
    pattern is universal. The interesting part is how it handles repositories
    with different structures and frontmatter conventions.

    Args:
        repo_owner: GitHub username or organization (e.g., 'OWASP')
        repo_name: Repository name

    Returns:
        List of dictionaries, each containing:
        - 'filename': str - path to file within repository
        - 'metadata': dict - YAML frontmatter key-value pairs (empty {} if no frontmatter)
        - 'content': str - markdown content after frontmatter (or full content if none)
    """
    # Download repository as zip
    zip_content = download_repo_zip(repo_owner, repo_name)

    # Open zip archive in memory
    with ZipFile(BytesIO(zip_content)) as zip_file:
        # Get all files
        all_files = zip_file.namelist()
        print(f"Total files in archive: {len(all_files)}")

        # Filter for markdown files
        markdown_files = [f for f in all_files if is_markdown_file(f)]
        print(f"Markdown files found: {len(markdown_files)}")

        # Process each markdown file
        documents = []
        for filename in markdown_files:
            file_content = zip_file.read(filename)
            parsed = parse_markdown_with_frontmatter(file_content)

            doc = {
                'filename': filename,
                'metadata': parsed['metadata'],
                'content': parsed['content']
            }
            documents.append(doc)

        print(f"Successfully processed {len(documents)} documents")
        return documents

## Testing with OWASP LLM Top 10 Repository

Repository: https://github.com/OWASP/www-project-top-10-for-large-language-model-applications

This repository documents the top 10 security risks for LLM applications. It has a different structure than our course examples and uses plain markdown (no frontmatter).

In [6]:
# Download and process OWASP repository
docs = read_repo_data("OWASP", "www-project-top-10-for-large-language-model-applications")

Downloaded 320788136 bytes
Total files in archive: 1076
Markdown files found: 542
Successfully processed 542 documents


In [7]:
# Display repository characteristics
print(f"OWASP LLM Top 10 Repository Analysis")
print(f"=" * 80)
print(f"Total documents: {len(docs)}")
print(f"Documents with frontmatter: {sum(1 for d in docs if d['metadata'])}")
print(f"Documents without frontmatter: {sum(1 for d in docs if not d['metadata'])}")
print()

# Show sample documents
print(f"Sample documents (first 3):")
for i, doc in enumerate(docs[:3], 1):
    print(f"\n{i}. {doc['filename']}")

    # Handle empty metadata gracefully
    if doc['metadata']:
        print(f"   Metadata: {doc['metadata']}")
    else:
        print(f"   Metadata: None (no frontmatter)")

    # Show content preview
    content_preview = doc['content'][:150].replace('\n', ' ')
    print(f"   Content preview: {content_preview}...")
    print(f"   Total length: {len(doc['content'])} characters")

# Compare to course repos
print(f"\n" + "=" * 80)
print(f"Comparison with Course Repositories:")
print(f"  DataTalks FAQ:     1,285 files, most with frontmatter")
print(f"  Evidently docs:       95 files, most with frontmatter")
print(f"  OWASP LLM Top 10:   {len(docs)} files, NO frontmatter")
print(f"\nKey insight: The same code works across all repos because python-frontmatter")
print(f"gracefully handles files without frontmatter (returns empty metadata dict).")

OWASP LLM Top 10 Repository Analysis
Total documents: 542
Documents with frontmatter: 3
Documents without frontmatter: 539

Sample documents (first 3):

1. www-project-top-10-for-large-language-model-applications-main/.github/PULL_REQUEST_TEMPLATE.md
   Metadata: None (no frontmatter)
   Content preview: # [Title of Your PR]  **Key Changes:**  - [ ] List major changes and core updates - [ ] Keep each line under 80 characters - [ ] Focus on the "what" a...
   Total length: 569 characters

2. www-project-top-10-for-large-language-model-applications-main/2_0_vulns/LLM00_Preface.md
   Metadata: None (no frontmatter)
   Content preview: ## Letter from the Project Leads  The OWASP Top 10 for Large Language Model Applications started in 2023 as a community-driven effort to highlight and...
   Total length: 3167 characters

3. www-project-top-10-for-large-language-model-applications-main/2_0_vulns/LLM01_PromptInjection.md
   Metadata: None (no frontmatter)
   Content preview: ## LLM01:2025 Pro

## What I Learned

**Repository Structure Adaptability:**
- The OWASP repository has 542 markdown files across nested directories (2_0_vulns/, documentation/, resources/)
- Unlike course repos that use frontmatter for metadata, OWASP files are plain markdown
- The `read_repo_data()` implementation handles this gracefully - no code changes needed

**Frontmatter Handling:**
- python-frontmatter library returns empty `metadata: {}` for files without frontmatter
- This makes the implementation robust - works with or without frontmatter
- Important to check `if doc['metadata']:` before displaying metadata fields to avoid confusing output

**Why This Matters for RAG:**
- Some documentation uses frontmatter (Jekyll, Hugo, Next.js sites) for structured metadata
- Other documentation is plain markdown with all structure in headers
- A good RAG ingestion pipeline must handle both cases
- Day 2 chunking will need to adapt strategy based on whether frontmatter provides structure

___

## Day 2: Chunking

Apply chunking strategies to OWASP documentation from Day 1. We'll implement sliding window chunking as the baseline, with full type annotations and validation.

In [8]:
# Day 2 imports
import copy
import tiktoken
import re
from typing import Any

In [9]:
def count_tokens(text: str, encoding: str = "cl100k_base") -> int:
    """Count tokens in text using tiktoken.

    Args:
        text: Text to count tokens for.
        encoding: Tiktoken encoding name. Use 'cl100k_base' for GPT-4/3.5-turbo
            and text-embedding-ada-002.

    Returns:
        Number of tokens in the text.

    Example:
        >>> count_tokens("Hello, world!")
        4
    """
    enc = tiktoken.get_encoding(encoding)
    return len(enc.encode(text))

In [10]:
def chunk_sliding_window(
    doc: dict[str, Any],
    chunk_size: int = 2000,
    overlap: int = 1000,
) -> list[dict[str, Any]]:
    """Split document content into overlapping chunks using sliding window.

    Implements a simple fixed-size sliding window algorithm. Each chunk overlaps
    with the next by `overlap` characters to preserve context at boundaries.

    Args:
        doc: Document dict with required keys:
            - 'content' (str): Document text to chunk
            - 'filename' (str): Source document identifier
            - 'metadata' (dict, optional): Frontmatter metadata to preserve
        chunk_size: Characters per chunk. Default 2000 (~512 tokens).
        overlap: Characters overlap between adjacent chunks. Default 1000.

    Returns:
        List of chunk dicts, each containing:
            - 'filename': Inherited from source document
            - 'metadata': Deep copy of source metadata
            - 'content': Chunk text
            - 'chunk_id': '{filename}-chunk-{index}' identifier
            - 'chunk_index': Zero-based position in chunk sequence
            - 'total_chunks': Total chunks from this document
            - 'chunk_method': 'sliding_window'

    Raises:
        ValueError: If doc missing required keys, or chunk_size <= overlap,
            or overlap < 0.

    Example:
        >>> doc = {'filename': 'test.md', 'metadata': {}, 'content': 'x' * 3000}
        >>> chunks = chunk_sliding_window(doc, chunk_size=2000, overlap=1000)
        >>> len(chunks)
        2
        >>> chunks[0]['chunk_id']
        'test.md-chunk-0'
    """
    # Input validation
    if "content" not in doc or "filename" not in doc:
        raise ValueError("Document must have 'content' and 'filename' keys")
    if chunk_size <= overlap:
        raise ValueError("chunk_size must be greater than overlap")
    if overlap < 0:
        raise ValueError("overlap must be non-negative")

    content = doc["content"]
    chunks: list[dict[str, Any]] = []
    start = 0

    while start < len(content):
        end = start + chunk_size
        chunk_content = content[start:end]

        chunk = {
            "filename": doc["filename"],
            "metadata": copy.deepcopy(doc.get("metadata", {})),
            "content": chunk_content,
            "chunk_id": f"{doc['filename']}-chunk-{len(chunks)}",
            "chunk_index": len(chunks),
            "total_chunks": -1,  # Placeholder, set after loop
            "chunk_method": "sliding_window",
        }
        chunks.append(chunk)
        start += chunk_size - overlap

    # Update total_chunks for all chunks
    total = len(chunks)
    for chunk in chunks:
        chunk["total_chunks"] = total

    return chunks

### Testing on OWASP Documents

Apply sliding window chunking to all OWASP documents from Day 1 and analyze the results.

In [11]:
# Chunk all OWASP documents
all_chunks: list[dict[str, Any]] = []
for doc in docs:
    chunks = chunk_sliding_window(doc, chunk_size=2000, overlap=1000)
    all_chunks.extend(chunks)

print(f"Total documents: {len(docs)}")
print(f"Total chunks: {len(all_chunks)}")
print(f"Average chunks per doc: {len(all_chunks) / len(docs):.1f}")

Total documents: 542
Total chunks: 3563
Average chunks per doc: 6.6


In [12]:
# Analyze first 3 chunks
for chunk in all_chunks[:3]:
    char_count = len(chunk["content"])
    token_count = count_tokens(chunk["content"])
    print(f"Chunk: {chunk['chunk_id']}")
    print(f"  Size: {char_count:,} chars, {token_count} tokens")
    print(f"  Token/char ratio: {token_count / char_count:.3f}")
    print(f"  Metadata keys: {list(chunk['metadata'].keys())}")
    print(f"  Position: {chunk['chunk_index'] + 1} of {chunk['total_chunks']}")
    print()

Chunk: www-project-top-10-for-large-language-model-applications-main/.github/PULL_REQUEST_TEMPLATE.md-chunk-0
  Size: 569 chars, 140 tokens
  Token/char ratio: 0.246
  Metadata keys: []
  Position: 1 of 1

Chunk: www-project-top-10-for-large-language-model-applications-main/2_0_vulns/LLM00_Preface.md-chunk-0
  Size: 2,000 chars, 369 tokens
  Token/char ratio: 0.184
  Metadata keys: []
  Position: 1 of 4

Chunk: www-project-top-10-for-large-language-model-applications-main/2_0_vulns/LLM00_Preface.md-chunk-1
  Size: 2,000 chars, 390 tokens
  Token/char ratio: 0.195
  Metadata keys: []
  Position: 2 of 4



In [13]:
# Verify metadata preservation and deep copy
if docs and all_chunks:
    original_doc = docs[0]
    first_chunk = all_chunks[0]

    # Check metadata preserved
    assert first_chunk["filename"] == original_doc["filename"], "Filename not preserved"
    assert first_chunk["metadata"] == original_doc.get("metadata", {}), "Metadata not preserved"

    # Check deep copy (modifying chunk doesn't affect original)
    first_chunk["metadata"]["_test_field"] = "test"
    assert "_test_field" not in original_doc.get("metadata", {}), "Deep copy failed"
    del first_chunk["metadata"]["_test_field"]

    print("Metadata preservation verified:")
    print(f"  - Filename preserved: {first_chunk['filename']}")
    print(f"  - Metadata deep copied (not shared reference)")
    print(f"  - Chunk fields added: chunk_id, chunk_index, total_chunks, chunk_method")

Metadata preservation verified:
  - Filename preserved: www-project-top-10-for-large-language-model-applications-main/.github/PULL_REQUEST_TEMPLATE.md
  - Metadata deep copied (not shared reference)
  - Chunk fields added: chunk_id, chunk_index, total_chunks, chunk_method


### Observations

*Document your analysis of the chunking results here:*

- How many chunks were created?
- What's the token/character ratio for OWASP content?
- Are there documents that produce many more chunks than average?
- Does the metadata preservation work correctly?

### Paragraph-Based Chunking

Paragraph chunking splits on blank lines, keeping each paragraph as an atomic unit. This respects natural writing structure but produces variable-sized chunks.

In [14]:
def chunk_by_paragraph(doc: dict[str, Any]) -> list[dict[str, Any]]:
    """Split document into paragraph chunks.

    Splits on blank lines (\\n\\s*\\n pattern), filters empty paragraphs,
    and preserves all metadata from the source document.

    Args:
        doc: Document dict with 'content', 'filename', and optional 'metadata' keys.

    Returns:
        List of chunk dicts, each containing:
        - filename: Inherited from source document
        - metadata: Deep copy of source metadata
        - content: Paragraph text (trimmed)
        - chunk_id: Format '{filename}-chunk-{index}'
        - chunk_index: Position in sequence (0-indexed)
        - total_chunks: Total paragraphs in document
        - chunk_method: 'paragraph'

    Raises:
        KeyError: If doc missing 'content' or 'filename' keys.

    Example:
        >>> doc = {'filename': 'test.md', 'metadata': {}, 'content': 'Para 1.\\n\\nPara 2.'}
        >>> chunks = chunk_by_paragraph(doc)
        >>> len(chunks)
        2
        >>> chunks[0]['chunk_method']
        'paragraph'
    """
    # Split on \n\s*\n pattern - matches paragraph boundaries
    paragraphs = re.split(r'\n\s*\n', doc['content'])

    # Filter empty and trim whitespace
    paragraphs = [p.strip() for p in paragraphs if p.strip()]

    # Create chunks with metadata preservation
    chunks = []
    for i, para_text in enumerate(paragraphs):
        chunk = {
            'filename': doc['filename'],
            'metadata': copy.deepcopy(doc.get('metadata', {})),
            'content': para_text,
            'chunk_id': f"{doc['filename']}-chunk-{i}",
            'chunk_index': i,
            'total_chunks': len(paragraphs),
            'chunk_method': 'paragraph'
        }
        chunks.append(chunk)

    return chunks

### Section-Based Chunking

Section chunking uses markdown structure (## headers) to find logical boundaries. Each section includes nested subsections until the next ## header.

In [15]:
def chunk_by_section(doc: dict[str, Any]) -> list[dict[str, Any]]:
    """Split document by ## markdown headers.

    Each chunk contains one ## section with all nested subsections (###, ####, etc.)
    until the next ## header. The header text is included in the chunk.

    Args:
        doc: Document dict with 'content', 'filename', and optional 'metadata' keys.

    Returns:
        List of chunk dicts with same structure as chunk_by_paragraph().
        If no ## headers found, returns single chunk with entire document.

    Raises:
        KeyError: If doc missing 'content' or 'filename' keys.

    Example:
        >>> doc = {'filename': 'test.md', 'metadata': {}, 'content': '## A\\nText\\n## B\\nMore'}
        >>> chunks = chunk_by_section(doc)
        >>> len(chunks)
        2
        >>> chunks[0]['content'].startswith('## A')
        True
    """
    # Find all ## headers and their positions
    pattern = r'^##\s+(.+)$'
    matches = list(re.finditer(pattern, doc['content'], re.MULTILINE))

    if not matches:
        # No sections found, return whole doc as single chunk
        return [{
            'filename': doc['filename'],
            'metadata': copy.deepcopy(doc.get('metadata', {})),
            'content': doc['content'],
            'chunk_id': f"{doc['filename']}-chunk-0",
            'chunk_index': 0,
            'total_chunks': 1,
            'chunk_method': 'section'
        }]

    chunks = []
    for i, match in enumerate(matches):
        start = match.start()
        # End is start of next section or end of document
        end = matches[i+1].start() if i+1 < len(matches) else len(doc['content'])

        section_text = doc['content'][start:end].strip()

        # Skip empty sections (header-only)
        if not section_text:
            continue

        chunk = {
            'filename': doc['filename'],
            'metadata': copy.deepcopy(doc.get('metadata', {})),
            'content': section_text,  # includes header
            'chunk_id': f"{doc['filename']}-chunk-{len(chunks)}",
            'chunk_index': len(chunks),
            'total_chunks': -1,  # Updated after loop
            'chunk_method': 'section'
        }
        chunks.append(chunk)

    # Update total_chunks for all
    total = len(chunks)
    for chunk in chunks:
        chunk['total_chunks'] = total

    return chunks

### Testing Paragraph and Section Chunking on OWASP Docs

Apply both strategies to the OWASP documents and compare chunk counts.

In [16]:
# Apply paragraph chunking to all OWASP docs
paragraph_chunks = []
for doc in docs:
    paragraph_chunks.extend(chunk_by_paragraph(doc))

print(f"Paragraph chunking: {len(docs)} docs -> {len(paragraph_chunks)} chunks")
print(f"Average: {len(paragraph_chunks) / len(docs):.1f} chunks per doc")

# Apply section chunking to all OWASP docs
section_chunks = []
for doc in docs:
    section_chunks.extend(chunk_by_section(doc))

print(f"\nSection chunking: {len(docs)} docs -> {len(section_chunks)} chunks")
print(f"Average: {len(section_chunks) / len(docs):.1f} chunks per doc")

Paragraph chunking: 542 docs -> 14254 chunks
Average: 26.3 chunks per doc

Section chunking: 542 docs -> 1023 chunks
Average: 1.9 chunks per doc


### Sample Chunks and Metadata Verification

Verify that chunks have correct structure and preserve metadata.

In [17]:
# Verify chunk structure
if paragraph_chunks:
    sample = paragraph_chunks[0]
    print("=== Paragraph Chunk Sample ===")
    print(f"chunk_id: {sample['chunk_id']}")
    print(f"chunk_index: {sample['chunk_index']} of {sample['total_chunks']}")
    print(f"chunk_method: {sample['chunk_method']}")
    print(f"filename: {sample['filename']}")
    print(f"metadata keys: {list(sample['metadata'].keys())}")
    print(f"content length: {len(sample['content'])} chars")

    # Assertions for correctness
    assert sample['chunk_method'] == 'paragraph', "chunk_method should be 'paragraph'"
    assert isinstance(sample['chunk_index'], int), "chunk_index should be int"
    assert sample['chunk_index'] < sample['total_chunks'], "chunk_index < total_chunks"
    print("Assertions passed!")
    print()

if section_chunks:
    sample = section_chunks[0]
    print("=== Section Chunk Sample ===")
    print(f"chunk_id: {sample['chunk_id']}")
    print(f"chunk_index: {sample['chunk_index']} of {sample['total_chunks']}")
    print(f"chunk_method: {sample['chunk_method']}")
    print(f"filename: {sample['filename']}")
    print(f"metadata keys: {list(sample['metadata'].keys())}")
    print(f"content length: {len(sample['content'])} chars")

    # Assertions for correctness
    assert sample['chunk_method'] == 'section', "chunk_method should be 'section'"
    assert isinstance(sample['chunk_index'], int), "chunk_index should be int"
    assert sample['chunk_index'] < sample['total_chunks'], "chunk_index < total_chunks"
    print("Assertions passed!")

=== Paragraph Chunk Sample ===
chunk_id: www-project-top-10-for-large-language-model-applications-main/.github/PULL_REQUEST_TEMPLATE.md-chunk-0
chunk_index: 0 of 11
chunk_method: paragraph
filename: www-project-top-10-for-large-language-model-applications-main/.github/PULL_REQUEST_TEMPLATE.md
metadata keys: []
content length: 20 chars
Assertions passed!

=== Section Chunk Sample ===
chunk_id: www-project-top-10-for-large-language-model-applications-main/.github/PULL_REQUEST_TEMPLATE.md-chunk-0
chunk_index: 0 of 1
chunk_method: section
filename: www-project-top-10-for-large-language-model-applications-main/.github/PULL_REQUEST_TEMPLATE.md
metadata keys: []
content length: 569 chars
Assertions passed!


### Verifying Nested Subsection Preservation

Per D-09, section chunks must include all nested subsections (###, ####) within each ## section. OWASP docs are heavily structured with nested headers - let's verify they're preserved.

In [18]:
# Verify nested subsections are preserved in section chunks (D-09)
nested_header_pattern = r'^###\s+.+$'

# Find section chunks that contain nested ### headers
chunks_with_nested = []
for chunk in section_chunks:
    nested_matches = re.findall(nested_header_pattern, chunk['content'], re.MULTILINE)
    if nested_matches:
        chunks_with_nested.append({
            'chunk_id': chunk['chunk_id'],
            'nested_headers': nested_matches[:3],  # Show first 3
            'nested_count': len(nested_matches)
        })

print(f"Section chunks containing nested ### headers: {len(chunks_with_nested)} of {len(section_chunks)}")
print()

# Show examples of preserved nested structure
if chunks_with_nested:
    print("Examples of nested subsection preservation:")
    for item in chunks_with_nested[:3]:
        print(f"\n  {item['chunk_id']}")
        print(f"    Nested headers ({item['nested_count']} total): {item['nested_headers']}")

# Assertion: If any doc has nested headers, at least some section chunks should preserve them
# (This confirms D-09 behavior - nested subsections stay with parent ## section)
docs_with_nested = sum(1 for doc in docs if re.search(r'^###\s+', doc['content'], re.MULTILINE))
if docs_with_nested > 0:
    assert len(chunks_with_nested) > 0, "D-09 violation: nested ### headers not found in any section chunk"
    print(f"\nD-09 VERIFIED: Nested subsections preserved in section chunks")
else:
    print("\nNote: No nested ### headers found in source docs (D-09 not applicable)")


Section chunks containing nested ### headers: 461 of 1023

Examples of nested subsection preservation:

  www-project-top-10-for-large-language-model-applications-main/2_0_vulns/LLM00_Preface.md-chunk-0
    Nested headers (2 total): ['### What’s New in the 2025 Top 10', '### Moving Forward']

  www-project-top-10-for-large-language-model-applications-main/2_0_vulns/LLM01_PromptInjection.md-chunk-0
    Nested headers (6 total): ['### Description', '### Types of Prompt Injection Vulnerabilities', '### Prevention and Mitigation Strategies']

  www-project-top-10-for-large-language-model-applications-main/2_0_vulns/LLM02_SensitiveInformationDisclosure.md-chunk-0
    Nested headers (4 total): ['### Description', '### Common Examples of Vulnerability', '### Prevention and Mitigation Strategies']

D-09 VERIFIED: Nested subsections preserved in section chunks
